In [33]:
import faiss

In [25]:
# import shutil
# import urllib.request as request
# from contextlib import closing

# # first we download the Sift1M dataset
# with closing(request.urlopen('ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz')) as r:
#     with open('sift.tar.gz', 'wb') as f:
#         shutil.copyfileobj(r, f)

In [ ]:
import numpy as np

# now define a function to read the fvecs file format of Sift1M dataset
def read_fvecs(fp):
    a = np.fromfile(fp, dtype='int32')
    d = a[0]
    return a.reshape(-1, d + 1)[:, 1:].copy().view('float32')

In [42]:
xb = read_fvecs("/export/kellywang/dataset/sift/sift/sift_base.fvecs")
print(xb.shape)

xq = read_fvecs("/export/kellywang/dataset/sift/sift/sift_query.fvecs")[0].reshape(1, -1)
print(xq.shape)


(1000000, 128)
(1, 128)


In [43]:
xb.dtype

dtype('float32')

In [52]:
nbits = 8
D = xb.shape[1]

m = 8
assert D % m == 0
index = faiss.IndexPQ(D, m, nbits)


In [53]:
index.train(xb)

In [54]:
index.add(xb)

In [56]:
k = 10
dist, I = index.search(xq, k)
print(dist, I)

[[58119.71  59916.15  59959.22  60811.47  62080.5   62911.453 64473.227
  66441.21  67029.66  67282.01 ]] [[932085 893601 561813  36267 670103 478814 721370 165581 872728 934876]]


In [62]:
import os
def get_memory(index):
    faiss.write_index(index, "./temp.index")
    file_size = os.path.getsize('./temp.index')
    os.remove('./temp.index')
    return file_size
get_memory(index)

8131158

In [60]:
%%timeit
index.search(xq,k)

5.63 ms ± 700 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [65]:
# IVF
# print(vecs)
nlist = 256
quantizer = faiss.IndexFlatL2(D)
index = faiss.IndexIVFPQ(quantizer, D, nlist    , m, nbits)
index.train(xb)
index.add(xb)
k = 10
dist, I = index.search(xq, k)
print(dist, I)





<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7335cc449380> >
[[62219.293 62537.2   63891.03  64203.016 65101.92  65498.555 66313.31
  66961.92  67586.086 67675.086]] [[377461 455537 779712 701258 562594 882943 600499 701919 562343 454263]]


[1 1 0 0 0]


In [ ]:
import numpy as np
a = np.random.rand(5, 100)
b = np.random.rand(2, 100)

a[:, None].shape, b[None, :].shape
c = np.linalg.norm(a[:, None] - b[None, :], axis=2)
c.shape
labels = np.argmin(c, axis=1)  
print(labels)

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.

In [18]:

def kmeans(vecs, k=3, max_iters=10, tol=1e-4):
    import numpy as np
    n_samples, _ = vecs.shape
    indices = np.random.choice(n_samples, k, replace=False)
    centroids = vecs[indices]

    for it in range(max_iters):
        # Compute distances to centroids
        distances = np.linalg.norm(vecs[:, None] - centroids[None, :], axis=2)
        labels = np.argmin(distances, axis=1)

        new_centroids = np.zeros_like(centroids)
        for cluster_idx in range(k):
            points = vecs[labels == cluster_idx]
            if len(points) > 0:
                new_centroids[cluster_idx] = points.mean(axis=0)
            else:
                # Handle edge case: Empty cluster. Keep old centroid.
                new_centroids[cluster_idx] = centroids[cluster_idx]

        diff = np.linalg.norm(new_centroids - centroids)
        if diff < tol:
            print(f"Converged at iteration {it}")
            return new_centroids, labels
        else:
            centroids = new_centroids

    return centroids, labels

In [20]:
# write test kmeans
vecs = np.random.rand(100, 128)
centroids, labels = kmeans(vecs, k=3, max_iters=10, tol=1e-4)
print(centroids)
print(labels)


Converged at iteration 2
[[0.49400051 0.38235951 0.55516111 0.58212063 0.52303249 0.53982302
  0.53344125 0.47096781 0.49799155 0.53637317 0.44497537 0.54407346
  0.50957024 0.59079055 0.33881384 0.53779822 0.29997193 0.45378898
  0.43398889 0.42195952 0.66592013 0.54535595 0.44403975 0.55064894
  0.51822435 0.49932832 0.48213092 0.44974066 0.63460101 0.63476984
  0.52573509 0.55543443 0.48908237 0.43482829 0.4825751  0.53360545
  0.51196156 0.48961837 0.32662938 0.45174873 0.50757454 0.52634561
  0.60464299 0.3987607  0.48678852 0.49310618 0.41446036 0.5607588
  0.4749978  0.62571063 0.42415958 0.3560771  0.36269317 0.42748771
  0.64466736 0.44788822 0.46769323 0.52324254 0.63567377 0.54845702
  0.3937788  0.57069194 0.52360734 0.55578342 0.5901497  0.46410263
  0.49541865 0.40946915 0.48647658 0.5406228  0.53087179 0.45651645
  0.47812932 0.64853764 0.5325248  0.49651365 0.75928331 0.55782262
  0.56693093 0.4513767  0.50118267 0.48566371 0.46710641 0.47158366
  0.44313511 0.5553635  